# Q Route: train the vehicle detector on a Roboflow dataset (Colab GPU)

This notebook trains a YOLOv8 vehicle detector on a free Colab GPU, using a
dataset you keep on Roboflow, and hands back a `best.pt` that the Q Route
backend can load.

Q Route's current detector (`results/yolo/dats_v8n/weights/best.pt`) was
trained on DATS_2022 on a laptop CPU: 25 epochs at 416 px took about 2 h 40 min.
A GPU makes longer runs at 640 px practical.

**Before you start**

1. **Runtime → Change runtime type → T4 GPU.**
2. Add your Roboflow API key in the **🔑 Secrets** panel (left sidebar) as
   `ROBOFLOW_API_KEY`, and switch on *Notebook access*. The key is read from
   there and never written into this notebook. If you skip this, cell 3 asks
   for it with hidden input.
3. **Check the dataset's licence.** Free Roboflow projects are public. Only
   upload data you are allowed to share, and only train on data whose licence
   allows it.

**The one step that matters most is class mapping (step 5).** The backend
counts vehicles by exact class name (`Bike`, `Car`, `Rikshaw`, …) and gives
each a PCU weight. A model whose classes are called `motorcycle` or
`auto-rickshaw` would load without error and count zero vehicles.

## 1. Check for a GPU

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not found")
assert torch.cuda.is_available(), (
    "No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install

In [ ]:
# The same major version the Q Route backend pins (backend/requirements.txt),
# so the best.pt this produces loads there without a format mismatch.
%pip install -q "ultralytics>=8.4,<9" roboflow

## 3. Roboflow API key

In [ ]:
import getpass

ROBOFLOW_API_KEY = None
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    pass  # no secret set, or notebook access not granted
if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass.getpass("Roboflow API key (input is hidden): ")

# Only whether it loaded, never the key itself.
print("API key loaded." if ROBOFLOW_API_KEY else "No API key - the download below will fail.")

## 4. Settings

`WORKSPACE`, `PROJECT` and `VERSION` come from the dataset's Roboflow page:
`universe.roboflow.com/<workspace>/<project>/dataset/<version>` (or your own
workspace's project page).

`yolov8n` is the same size as the model Q Route serves now. The backend runs
detection on a CPU, so a bigger model (`yolov8s.pt`) is more accurate but
slows down every camera analysis.

In [ ]:
WORKSPACE = ""        # e.g. "my-workspace"
PROJECT   = ""        # e.g. "indian-vehicles"
VERSION   = 1         # the dataset version number on Roboflow

BASE_MODEL = "yolov8n.pt"   # COCO-pretrained starting point, as before
EPOCHS     = 60             # an upper bound; PATIENCE stops earlier if val mAP stalls
IMGSZ      = 640
BATCH      = 16
PATIENCE   = 15
RUN_NAME   = "roboflow_v8n"

assert WORKSPACE and PROJECT, "Fill in WORKSPACE and PROJECT above, then run this cell again."

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
dataset = version.download("yolov8", location="/content/source_dataset", overwrite=True)
SRC = dataset.location
print("Downloaded to", SRC)

## 5. Map the dataset's classes onto Q Route's

The trained model must use Q Route's 12 class names, **in Q Route's order**:
the backend reads names, and keeping the order also means your current model
and the new one can be scored against each other on the same images (step 8).

The cell below guesses a mapping from the names and a table of common aliases.
Anything it cannot place is **dropped** (its boxes are removed from training).
Read the printout, and fix anything wrong in the cell after it.

In [ ]:
import pathlib
import re

import yaml

# Must match backend/vision/detector.py (VEHICLE_PCU + NON_VEHICLE) and the
# class order of the current best.pt.
QROUTE_CLASSES = ["Bike", "Car", "Rikshaw", "Tempo", "Bus", "Truck", "Cycle", "Cart",
                  "person", "Traffic Signal", "Zebra Crossing", "Traffic Sign Board"]

# Keys are lower-case letters only ("Auto-Rickshaw" -> "autorickshaw").
ALIASES = {
    "motorcycle": "Bike", "motorbike": "Bike", "twowheeler": "Bike", "scooter": "Bike",
    "scooty": "Bike", "moped": "Bike",
    "bicycle": "Cycle",
    "autorickshaw": "Rikshaw", "auto": "Rikshaw", "rickshaw": "Rikshaw",
    "threewheeler": "Rikshaw", "erickshaw": "Rikshaw",
    "suv": "Car", "sedan": "Car", "hatchback": "Car", "taxi": "Car", "jeep": "Car",
    "minibus": "Bus",
    "lorry": "Truck",
    "lcv": "Tempo", "minitruck": "Tempo", "pickup": "Tempo",
    "handcart": "Cart", "animalcart": "Cart", "bullockcart": "Cart",
    "pedestrian": "person", "people": "person",
    "trafficlight": "Traffic Signal",
    "crosswalk": "Zebra Crossing",
    "trafficsign": "Traffic Sign Board", "signboard": "Traffic Sign Board",
}


def norm(s):
    return re.sub(r"[^a-z]", "", str(s).lower())


src_spec = yaml.safe_load(open(f"{SRC}/data.yaml", encoding="utf-8"))
src_names = src_spec["names"]
if isinstance(src_names, dict):
    src_names = [src_names[k] for k in sorted(src_names)]

by_norm = {norm(q): q for q in QROUTE_CLASSES}
CLASS_MAP = {name: by_norm.get(norm(name)) or ALIASES.get(norm(name)) for name in src_names}

print(f"{len(src_names)} classes in the dataset:\n")
for name, target in CLASS_MAP.items():
    print(f"  {name!r:30} -> {target or 'DROPPED'}")

In [ ]:
# Fix the mapping here if the guess is wrong. Examples:
#   CLASS_MAP["van"] = "Car"          # a passenger van counts like a car (1.0 PCU)
#   CLASS_MAP["tractor"] = "Truck"
#   CLASS_MAP["animal"] = None        # drop it
# Valid targets: QROUTE_CLASSES, or None to drop.

bad = {k: v for k, v in CLASS_MAP.items() if v is not None and v not in QROUTE_CLASSES}
assert not bad, f"Not a Q Route class: {bad}"
assert any(v in QROUTE_CLASSES[:8] for v in CLASS_MAP.values()), "No vehicle class is mapped."

## 6. Build the remapped dataset

Rewrites every label file into Q Route's class indices and reports how many
boxes each class has. A class with very few boxes cannot be learned: in
DATS_2022, Cycle (43 boxes) and Cart (4) ended up never detected.

In [ ]:
import collections
import os
import shutil

DST = pathlib.Path("/content/qroute_dataset")
shutil.rmtree(DST, ignore_errors=True)

index_of = {q: i for i, q in enumerate(QROUTE_CLASSES)}
remap = {i: (index_of[CLASS_MAP[n]] if CLASS_MAP.get(n) else None) for i, n in enumerate(src_names)}

instances, images, dropped = {}, {}, collections.Counter()
for split in ("train", "valid", "test"):
    img_dir = pathlib.Path(SRC) / split / "images"
    lbl_dir = pathlib.Path(SRC) / split / "labels"
    if not img_dir.exists():
        continue
    (DST / split / "images").mkdir(parents=True)
    (DST / split / "labels").mkdir(parents=True)
    tally, n_img = collections.Counter(), 0
    for img in sorted(img_dir.iterdir()):
        target = DST / split / "images" / img.name
        try:
            os.link(img, target)          # same disk: no second copy of the images
        except OSError:
            shutil.copy2(img, target)
        n_img += 1
        out = []
        label = lbl_dir / (img.stem + ".txt")
        if label.exists():
            for line in label.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                new = remap.get(int(parts[0]))
                if new is None:
                    dropped[src_names[int(parts[0])]] += 1
                    continue
                out.append(" ".join([str(new)] + parts[1:]))
                tally[QROUTE_CLASSES[new]] += 1
        (DST / split / "labels" / (img.stem + ".txt")).write_text("\n".join(out) + ("\n" if out else ""))
    instances[split], images[split] = tally, n_img

assert "train" in images and "valid" in images, "The dataset needs both a train and a valid split."

data = {"path": str(DST), "train": "train/images", "val": "valid/images", "names": QROUTE_CLASSES}
if "test" in images:
    data["test"] = "test/images"
DATA_YAML = str(DST / "data.yaml")
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("images:", images)
print(f"\n{'class':20} {'train':>7} {'valid':>7}")
for q in QROUTE_CLASSES:
    tr, va = instances["train"][q], instances["valid"][q]
    note = "   <- too few to learn" if 0 < tr < 50 else ("   (none)" if tr == 0 else "")
    print(f"{q:20} {tr:7} {va:7}{note}")
if dropped:
    print("\ndropped boxes:", dict(dropped))

## 7. Train

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    project="/content/runs",
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    plots=True,
)
SAVE_DIR = pathlib.Path(model.trainer.save_dir)
BEST = SAVE_DIR / "weights" / "best.pt"
print("best weights:", BEST)

## 8. Score it

These numbers are measured on **this dataset's validation images**. They are
not comparable with the 0.40 mAP50 in HANDOFF.md, which was measured on
DATS_2022. The validation set also chose which epoch became `best.pt`, so it
flatters the model slightly.

Q Route runs detection at **416 px**, so the model is scored at both 416 and
the training size.

In [ ]:
def score(weights, label):
    m = YOLO(str(weights))
    rows = {}
    for size in sorted({416, IMGSZ}):
        r = m.val(data=DATA_YAML, imgsz=size, device=0, plots=False, verbose=False)
        rows[size] = {"mAP50": round(float(r.box.map50), 3), "mAP50-95": round(float(r.box.map), 3),
                      "precision": round(float(r.box.mp), 3), "recall": round(float(r.box.mr), 3)}
        print(f"{label:24} @{size}px  " + "  ".join(f"{k} {v}" for k, v in rows[size].items()))
    last = r
    per_class = {m.names[int(c)]: round(float(last.box.ap50[i]), 3) for i, c in enumerate(last.box.ap_class_index)}
    return rows, per_class

NEW_SCORES, NEW_PER_CLASS = score(BEST, "new model")
print("\nmAP50 per class (training size):")
for name in QROUTE_CLASSES[:8]:
    print(f"  {name:10} {NEW_PER_CLASS.get(name, 'no val boxes')}")

### Optional: score your current model on the same images

Upload `results/yolo/dats_v8n/weights/best.pt` from your Q Route folder. Both
models then face the same validation images, which is the only fair
comparison. Skip this cell if you do not need it.

In [ ]:
from google.colab import files

print("Choose results/yolo/dats_v8n/weights/best.pt from your Q Route folder.")
uploaded = files.upload()
OLD = pathlib.Path(next(iter(uploaded)))
old_names = list(YOLO(str(OLD)).names.values())
assert old_names == QROUTE_CLASSES, f"Class order differs, so the scores would not line up: {old_names}"
OLD_SCORES, _ = score(OLD, "current Q Route model")

## 9. Download the result

In [ ]:
import json

from google.colab import files

OUT = pathlib.Path(f"/content/{RUN_NAME}")
shutil.rmtree(OUT, ignore_errors=True)
(OUT / "weights").mkdir(parents=True)
shutil.copy2(BEST, OUT / "weights" / "best.pt")
for name in ("results.csv", "args.yaml", "results.png", "confusion_matrix.png"):
    if (SAVE_DIR / name).exists():
        shutil.copy2(SAVE_DIR / name, OUT / name)

report = {
    "source": f"roboflow:{WORKSPACE}/{PROJECT}/{VERSION}",
    "base_model": BASE_MODEL, "epochs_requested": EPOCHS, "imgsz": IMGSZ,
    "class_map": CLASS_MAP, "images": images,
    "instances": {s: dict(c) for s, c in instances.items()},
    "val_scores": NEW_SCORES, "val_map50_per_class": NEW_PER_CLASS,
    "current_model_val_scores": globals().get("OLD_SCORES"),
    "note": "Scores are on this dataset's validation split, which also selected best.pt.",
}
(OUT / "training_report.json").write_text(json.dumps(report, indent=2))

archive = shutil.make_archive(str(OUT), "zip", OUT)
files.download(archive)

## 10. Put it into Q Route

1. Unzip into `results/yolo/roboflow_v8n/` in your Q Route folder, so the
   weights sit at `results/yolo/roboflow_v8n/weights/best.pt`. Keep the old
   `dats_v8n` folder.
2. The backend loads the path in `WEIGHTS` in `backend/vision/detector.py`
   (`results/yolo/dats_v8n/weights/best.pt`). Point it at the new file.
3. It detects at `imgsz=416` (`RoadVisionAnalyser` in the same file). If step 8
   shows a clear gain at 640, raise it. Each analysis on the CPU server then
   takes roughly twice as long or more.
4. Restart the backend.
5. Run `backend/scripts/calibrate_yolo.py` to measure the counting error of the
   detector now being served, and update HANDOFF.md's figures only from that run.